# Step 4: SQL-Based Business Analysis
Ingests cleaned data into SQL database and performs queries using Joins, Aggregations, CTEs, Window Functions, and Subqueries.


In [ ]:
import sqlite3
import os
import sys
import os

def resolve_path(rel_path):
    curr = os.path.abspath(os.getcwd())
    while curr and os.path.dirname(curr) != curr:
        candidate = os.path.join(curr, rel_path)
        if os.path.exists(candidate):
            return os.path.abspath(candidate)
        curr = os.path.dirname(curr)
    return os.path.abspath(rel_path)
import pandas as pd

db_path = resolve_path("data/cleaned/ecommerce.db")
print(f"Connected to database: {db_path}")
conn = sqlite3.connect(db_path)
conn.execute("PRAGMA temp_store = MEMORY;")

query_cte = '''
WITH CustomerSpend AS (
    SELECT 
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS order_count,
        SUM(oi.price + oi.freight_value) AS total_spent
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY c.customer_unique_id
)
SELECT 
    customer_unique_id,
    order_count,
    ROUND(total_spent, 2) AS total_spent,
    RANK() OVER (ORDER BY total_spent DESC) AS spending_rank
FROM CustomerSpend
LIMIT 10;
'''

df = pd.read_sql_query(query_cte, conn)
print(df, flush=True)
conn.close()

